In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# Step 1: Full load from bronze_transactions as silver baseline
# Normalize columns to match the unified silver schema

bronze_txns = spark.read.table("fintech_fraud_risk.bronze.bronze_transactions")

silver_base = bronze_txns.select(
    F.col("transaction_id").cast("int").alias("transaction_id"),
    F.col("customer_id").cast("int").alias("customer_id"),
    F.col("merchant_id").cast("int").alias("merchant_id"),
    F.col("transaction_timestamp").cast("timestamp").alias("transaction_timestamp"),
    F.col("amount").cast("double").alias("amount"),
    F.col("currency"),
    F.col("payment_method"),
    F.col("transaction_type"),
    F.col("transaction_status"),
    F.col("device_id").cast("int").alias("device_id"),
    F.col("location"),
    F.col("source_system"),
    F.col("file_name").alias("_source_file"),
    F.lit(None).cast("timestamp").alias("_ingested_at"),
)

# Write as silver table (overwrite on first run; use append on subsequent runs)
(silver_base.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("fintech_fraud_risk.silver.transactions"))

print(f"Silver baseline loaded: {silver_base.count()} rows from bronze_transactions")

In [0]:
# Step 2: MERGE CDC incremental data into silver.transactions
# Using SQL MERGE for Spark Connect compatibility

spark.sql("""
MERGE INTO fintech_fraud_risk.silver.transactions AS target
USING (
    SELECT
        CAST(transaction_id AS INT)       AS transaction_id,
        CAST(customer_id AS INT)           AS customer_id,
        CAST(merchant_id AS INT)           AS merchant_id,
        CAST(transaction_timestamp AS TIMESTAMP) AS transaction_timestamp,
        CAST(amount AS DOUBLE)             AS amount,
        currency,
        payment_method,
        transaction_type,
        transaction_status,
        CAST(device_id AS INT)             AS device_id,
        location,
        source_system,
        _source_file,
        _ingested_at
    FROM fintech_fraud_risk.bronze.bronze_transaction_cdc
) AS source
ON target.transaction_id = source.transaction_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

cdc_count = spark.read.table("fintech_fraud_risk.bronze.bronze_transaction_cdc").count()
print(f"MERGE complete: {cdc_count} CDC rows merged into silver.transactions")

In [0]:
# Step 3: Verify silver layer

result = spark.read.table("fintech_fraud_risk.silver.transactions")
print(f"Total silver rows: {result.count()}")
print(f"Distinct transaction_ids: {result.select('transaction_id').distinct().count()}")
print(f"ID range: {result.agg(F.min('transaction_id'), F.max('transaction_id')).collect()[0]}")
result.printSchema()
display(result.orderBy(F.col("transaction_id").desc()).limit(10))